Checagem 1 — Consistência entre pessoas e a soma de mortos/feridos/ilesos/ignorados

In [0]:
df_silver = spark.table("prf_acidentes.silver.acidentes_limpo")
fato_acidente = spark.table("prf_acidentes.gold.fato_acidente")

print("Linhas Silver:", df_silver.count())
print("Linhas Gold (fato_acidente):", fato_acidente.count())

In [0]:
from pyspark.sql import functions as F

df_check1 = (
    df_silver
    .withColumn(
        "soma_calculada",
        F.col("mortos") + F.col("feridos_leves") + F.col("feridos_graves") + F.col("ilesos") + F.col("ignorados")
    )
    .withColumn("diferenca", F.col("pessoas") - F.col("soma_calculada"))
)

# Quantas linhas batem exatamente?
total = df_check1.count()
batem = df_check1.filter(F.col("diferenca") == 0).count()
nao_batem = total - batem

print(f"Total de linhas: {total}")
print(f"Linhas onde pessoas = soma dos estados: {batem}")
print(f"Linhas com divergência: {nao_batem}")

# Mostrar alguns exemplos de divergência, se houver
df_check1.filter(F.col("diferenca") != 0).select(
    "id", "pessoas", "mortos", "feridos_leves", "feridos_graves", "ilesos", "ignorados", "soma_calculada", "diferenca"
).show(20)

In [0]:
from pyspark.sql import functions as F

# Testar se a divergência segue o padrão: diferenca = 1 - ignorados
df_check1_padrao = df_check1.withColumn(
    "segue_padrao", F.col("diferenca") == (1 - F.col("ignorados"))
)

total_divergentes = df_check1_padrao.filter(F.col("diferenca") != 0).count()
seguem_padrao = df_check1_padrao.filter((F.col("diferenca") != 0) & (F.col("segue_padrao") == True)).count()

print(f"Total de linhas divergentes: {total_divergentes}")
print(f"Linhas que seguem o padrão (diferença = 1 - ignorados): {seguem_padrao}")

In [0]:
from pyspark.sql import functions as F

# Isolar as linhas divergentes que NÃO seguem o padrão "diferenca = 1 - ignorados"
df_resto = df_check1_padrao.filter((F.col("diferenca") != 0) & (F.col("segue_padrao") == False))

print("Total de linhas no grupo residual:", df_resto.count())

df_resto.select(
    "id", "pessoas", "mortos", "feridos_leves", "feridos_graves", "ilesos", "ignorados", "soma_calculada", "diferenca"
).show(20)

# Ver a distribuição dos valores de "diferenca" nesse grupo residual
df_resto.groupBy("diferenca").count().orderBy("diferenca").show(30)

Checagem 2 — Validação de domínio: uf e br

In [0]:
from pyspark.sql import functions as F

# UFs válidas do Brasil
ufs_validas = ["AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"]

df_silver.groupBy("uf").count().orderBy(F.desc("count")).show(30)

print("--- BR fora do intervalo plausível (ex: <1 ou >495) ---")
df_silver.filter((F.col("br") < 1) | (F.col("br") > 495)).select("br").distinct().show()

In [0]:
from pyspark.sql import functions as F

df_silver.filter(F.col("br") == 0).select("id", "uf", "br", "km", "municipio").show(10)
print("Total com br = 0:", df_silver.filter(F.col("br") == 0).count())

df_silver.filter(F.col("br") == 498).select("id", "uf", "br", "km", "municipio").show(10)
print("Total com br = 498:", df_silver.filter(F.col("br") == 498).count())

Checagem 3 — Verificar duplicatas na Gold (fato_acidente) após os joins

In [0]:
total_fato = fato_acidente.count()
ids_distintos_fato = fato_acidente.select("id_acidente").distinct().count()

print(f"Total de linhas na fato_acidente: {total_fato}")
print(f"IDs de acidente distintos: {ids_distintos_fato}")
print(f"Diferença: {total_fato - ids_distintos_fato}")